Evaluate A2C Ab Study 3

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari
!ls -la

/content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari
total 41
drwx------ 2 root root  4096 Sep 25 15:17 code
drwx------ 2 root root  4096 Sep 25 15:04 .git
-rw------- 1 root root 13586 Oct 31 06:01 github_terminal.ipynb
-rw------- 1 root root    33 Sep 26 19:25 .gitignore
drwx------ 2 root root  4096 Sep 25 15:17 models
drwx------ 2 root root  4096 Oct 31 02:25 OLD
-rw------- 1 root root  2348 Sep 29 02:21 README.md
drwx------ 2 root root  4096 Sep 25 15:17 results
drwx------ 2 root root  4096 Oct 18 05:10 videos


In [3]:
!pip install stable-baselines3 gymnasium[atari,accept-rom-license] ale-py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 8.2 MB/s eta 0:00:00


In [4]:
# from stable_baselines3.common.evaluation import evaluate_policy
# evaluate_policy basically does what my manual for loop does
# evaluate_policy(model, env, n_eval_episodes=10, deterministic=True, render=False, warn=False)
# NOTE: I don't use this bc I need individual rewards for each episode for calculating HNS and HWRNS later

import os
import torch
import gymnasium as gym
import stable_baselines3
import ale_py
import numpy as np
import random

# Algorithm
from stable_baselines3 import A2C

# For debugging
from stable_baselines3.common.monitor import Monitor
import time

# Action masking
from stable_baselines3.common.atari_wrappers import AtariWrapper

# Vector environment
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, DummyVecEnv

# Visualization
import moviepy.editor as mpy
from IPython.display import HTML
from base64 import b64encode
import matplotlib.pyplot as plt

print("All imports working")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
  return datetime.utcnow().replace(tzinfo=utc)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294

All imports working


In [5]:
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: Tesla T4


Create Environment

In [6]:
game_name = "ALE/Bowling-v5"

In [7]:
seed = 80
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [8]:
def make_env(render_mode=None):
  env = gym.make("ALE/Bowling-v5", render_mode=render_mode)

  # disable reward clipping
  env = AtariWrapper(env, clip_reward=False)

  # Monitor should wrap it last. Gives the Mean Episode Length & Reward
  env = Monitor(env)

  return env

In [9]:
action_dict = {
  0: "NOOP",
  1: "FIRE",
  2: "UP",
  3: "DOWN",
  4: "UPFIRE",
  5: "DOWNFIRE"
}

Load Model

In [10]:
# Current working directory:
# /content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari

# Load model
model_name = "a2c_10000000_Ablation_Study_3"

Video

In [11]:
video_env = DummyVecEnv([lambda:make_env("rgb_array")])
video_env.seed(seed)
video_env = VecFrameStack(video_env, n_stack=4)

In [12]:
# Load model with video_env
model = A2C.load(
    f"models/{model_name}",
    env=video_env,
    device="cuda"
)

print("Model loaded")

Wrapping the env in a VecTransposeImage.


  return datetime.utcnow().replace(tzinfo=utc)



Model loaded


In [13]:
def evaluate_model_and_make_video(model, n_eval_episodes, env, video_path):
  """Evaluate model and return mean reward"""

  frames = []

  all_rewards = []
  all_steps = []
  for episode in range(n_eval_episodes):
    obs = env.reset()
    episode_reward = 0
    dones = [False]
    steps = 0

    while not dones[0]:
      # Predict next action
      action, _ = model.predict(obs, deterministic=True)
      obs, rewards, dones, infos = env.step(action)

      # Get a frame from the first environment inside the VecEnv
      frame = env.envs[0].render()
      frames.append(frame)

      episode_reward += rewards[0]
      steps += 1
      if rewards > 0:
        print(f"Step {steps}: Action:{action_dict[action[0]]}   | Reward: {rewards[0]}   | total reward: {episode_reward}")
      # print(done)
      # Actions: NOOP(0), FIRE(1), UP(2), DOWN(3), UPFIRE(4), DOWNFIRE(5)
      # print(f"Reward earned for doing {action_dict[action[0]]}: {reward[0]}")

    print(f"Episode {episode+1}: Reward = {episode_reward:6.1f}, Steps = {steps}")
    all_rewards.append(episode_reward)
    all_steps.append(steps)

  # Save to mp4
  clip = mpy.ImageSequenceClip(frames, fps=30)
  clip.write_videofile(video_save_path)

  return all_rewards, all_steps

In [14]:
# n_eval_episodes = 3
n_eval_episodes = 1
video_name = f"xxx_playing_around_with_A2C_Ablation_Study_3_{n_eval_episodes}_episodes.mp4"
video_save_path = f"/content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari/videos/{video_name}"

In [15]:
all_rewards, all_steps = evaluate_model_and_make_video(model, n_eval_episodes, video_env, video_name)

Step 51: Action:FIRE   | Reward: 9.0   | total reward: 9.0
Step 103: Action:FIRE   | Reward: 9.0   | total reward: 18.0
Step 181: Action:FIRE   | Reward: 19.0   | total reward: 37.0
Step 206: Action:FIRE   | Reward: 9.0   | total reward: 46.0
Step 258: Action:FIRE   | Reward: 9.0   | total reward: 55.0
Step 336: Action:FIRE   | Reward: 19.0   | total reward: 74.0
Step 361: Action:FIRE   | Reward: 9.0   | total reward: 83.0
Step 413: Action:FIRE   | Reward: 9.0   | total reward: 92.0
Step 491: Action:FIRE   | Reward: 19.0   | total reward: 111.0
Step 516: Action:FIRE   | Reward: 9.0   | total reward: 120.0
Episode 1: Reward =  120.0, Steps = 516
Moviepy - Building video /content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari/videos/xxx_playing_around_with_A2C_Ablation_Study_3_1_episodes.mp4.
Moviepy - Writing video /content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari/videos/xxx_playing_around_with_A2C_Ablation_Study_3_1_episodes.mp4



Moviepy - Done !
Moviepy - video ready /content/drive/MyDrive/MECE689_Bowling/MECE689_RL_Bowling_Atari/videos/xxx_playing_around_with_A2C_Ablation_Study_3_1_episodes.mp4


In [16]:
mean_reward = np.mean(all_rewards)
mean_steps = np.mean(all_steps)
print(f"{mean_reward:.4f} mean reward, {mean_steps:.4f} mean steps")

120.0000 mean reward, 516.0000 mean steps


In [17]:
# Display the video inline as part of Google Colab
mp4 = open(video_save_path, "rb").read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML(f'<video width=480 controls><source src="{data_url}" type="video/mp4"></video>')

In [18]:
video_env.close()
del video_env

In [19]:
del model

Chunk into Bowling Frames

In [20]:
video_env = DummyVecEnv([lambda:make_env("rgb_array")])
video_env.seed(seed)
video_env = VecFrameStack(video_env, n_stack=4)

In [21]:
# Load model with video_env
model = A2C.load(
    f"models/{model_name}",
    env=video_env,
    device="cuda"
)

print("Model loaded")

Wrapping the env in a VecTransposeImage.
Model loaded


In [22]:
def evaluate_model_and_make_video(model, n_eval_episodes, env):
  """Evaluate model and return mean reward"""

  frames = []
  all_rewards = []
  all_steps = []

  for episode in range(n_eval_episodes):
    obs = env.reset()
    episode_reward = 0
    dones = [False]
    steps = 0

    # Track bowling frames
    bowling_frame = 1
    frame_start_step = 0
    frame_start_score = 0
    last_reward_step = 0

    print(f"EPISODE {episode + 1}\n")

    while not dones[0]:
      action, _ = model.predict(obs, deterministic=True)
      obs, rewards, dones, infos = env.step(action)

      frame = env.envs[0].render()
      frames.append(frame)

      episode_reward += rewards[0]
      steps += 1

      # Detect when we get a reward (pins knocked down or scoring updated)
      if rewards[0] > 0:
        # Calculate frame statistics
        steps_in_frame = steps - frame_start_step
        frame_score = rewards[0]

        print(f"BOWLING FRAME {bowling_frame} COMPLETE")
        print(f"Score this frame: {frame_score}")
        print(f"Total score: {episode_reward}")
        print(f"Steps in frame: {steps_in_frame} (steps {frame_start_step+1}-{steps})")
        print(f"Last action: {action_dict[action[0]]}\n\n")

        # Update for next frame
        bowling_frame += 1
        frame_start_step = steps
        frame_start_score = episode_reward
        last_reward_step = steps

      if dones[0]:
        # CAPTURE ADDITIONAL FRAMES AFTER EPISODE ENDS
        # b/c game ends at 179, before it adds 9 to the score to get the final score of 188
        # Capture 50 more frames
        for _ in range(100):
          frame = env.envs[0].render()
          frames.append(frame)

    print(f"EPISODE {episode+1} SUMMARY: Total Reward = {episode_reward:6.1f}, Total Steps = {steps}")

    all_rewards.append(episode_reward)
    all_steps.append(steps)

  return all_rewards, all_steps

In [23]:
n_eval_episodes = 1

In [24]:
all_rewards, all_steps = evaluate_model_and_make_video(model, n_eval_episodes, video_env)

EPISODE 1

BOWLING FRAME 1 COMPLETE
Score this frame: 9.0
Total score: 9.0
Steps in frame: 51 (steps 1-51)
Last action: FIRE


BOWLING FRAME 2 COMPLETE
Score this frame: 9.0
Total score: 18.0
Steps in frame: 52 (steps 52-103)
Last action: FIRE


BOWLING FRAME 3 COMPLETE
Score this frame: 19.0
Total score: 37.0
Steps in frame: 78 (steps 104-181)
Last action: FIRE


BOWLING FRAME 4 COMPLETE
Score this frame: 9.0
Total score: 46.0
Steps in frame: 25 (steps 182-206)
Last action: FIRE


BOWLING FRAME 5 COMPLETE
Score this frame: 9.0
Total score: 55.0
Steps in frame: 52 (steps 207-258)
Last action: FIRE


BOWLING FRAME 6 COMPLETE
Score this frame: 19.0
Total score: 74.0
Steps in frame: 78 (steps 259-336)
Last action: FIRE


BOWLING FRAME 7 COMPLETE
Score this frame: 9.0
Total score: 83.0
Steps in frame: 25 (steps 337-361)
Last action: FIRE


BOWLING FRAME 8 COMPLETE
Score this frame: 9.0
Total score: 92.0
Steps in frame: 52 (steps 362-413)
Last action: FIRE


BOWLING FRAME 9 COMPLETE
Score t

In [25]:
mean_reward = np.mean(all_rewards)
mean_steps = np.mean(all_steps)
print(f"{mean_reward:.4f} mean reward, {mean_steps:.4f} mean steps")

120.0000 mean reward, 516.0000 mean steps


In [26]:
video_env.close()
del video_env

In [27]:
del model